In [1]:
# 02_preprocessing.ipynb

# %% [markdown]
# # Regulatory-Constrained Fairness Framework for Credit Scoring
# ## 02. Preprocessing
#
# Prepare each credit dataset for modeling and fairness evaluation:
# - numeric imputation (median) + scaling
# - categorical imputation ("missing" category) + one-hot encoding
# - stratified train/test split
# - persist processed splits and a shared metadata dict for later notebooks
#
# Sensitive attributes are kept as separate columns (not scaled/encoded away) so that
# AIF360 / Fairlearn can consume them downstream.

# %%
# Imports and configuration
import os
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

DATA_DIR = os.path.join("..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")

# Global config
RANDOM_STATE = 42
TEST_SIZE = 0.25

os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

# %%
# Load frames produced by 01_data_loading
df_german = pd.read_parquet(os.path.join(PROCESSED_DIR, "german.parquet"))
df_taiwan = pd.read_parquet(os.path.join(PROCESSED_DIR, "taiwan.parquet"))
df_home = pd.read_parquet(os.path.join(PROCESSED_DIR, "credit_card_approval.parquet"))

print("german:", df_german.shape)
print("taiwan:", df_taiwan.shape)
print("credit_card_approval:", df_home.shape)

# %%
# Shared preprocessing helper.
# Splits first, then fits imputation/scaling/encoding on train only to avoid leakage.
def preprocess_dataset(
    df, target_col, sensitive_cols, drop_cols=None,
    test_size=TEST_SIZE, random_state=RANDOM_STATE
):
    df = df.copy()
    drop_cols = drop_cols or []

    # Separate label and sensitive attributes; the rest are model features
    y = df[target_col].astype(int)
    sensitive = df[sensitive_cols].copy()

    feature_df = df.drop(columns=[target_col] + sensitive_cols + drop_cols, errors="ignore")

    # Identify column types on the feature matrix
    num_cols = feature_df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = feature_df.select_dtypes(exclude=[np.number]).columns.tolist()

    # Stratified split on the label (indices shared across features + sensitive)
    idx_train, idx_test = train_test_split(
        feature_df.index, test_size=test_size,
        random_state=random_state, stratify=y
    )

    X_train_raw = feature_df.loc[idx_train].copy()
    X_test_raw = feature_df.loc[idx_test].copy()

    # --- Numeric: median imputation (fit on train) + standard scaling ---
    num_medians = X_train_raw[num_cols].median()
    X_train_raw[num_cols] = X_train_raw[num_cols].fillna(num_medians)
    X_test_raw[num_cols] = X_test_raw[num_cols].fillna(num_medians)

    scaler = StandardScaler()
    X_train_num = pd.DataFrame(
        scaler.fit_transform(X_train_raw[num_cols]),
        columns=num_cols, index=X_train_raw.index
    ) if num_cols else pd.DataFrame(index=X_train_raw.index)
    X_test_num = pd.DataFrame(
        scaler.transform(X_test_raw[num_cols]),
        columns=num_cols, index=X_test_raw.index
    ) if num_cols else pd.DataFrame(index=X_test_raw.index)

    # --- Categorical: fill missing with "missing", one-hot (align train/test) ---
    if cat_cols:
        X_train_cat = X_train_raw[cat_cols].astype("object").fillna("missing")
        X_test_cat = X_test_raw[cat_cols].astype("object").fillna("missing")
        X_train_cat = pd.get_dummies(X_train_cat, drop_first=False)
        X_test_cat = pd.get_dummies(X_test_cat, drop_first=False)
        # Align columns so train and test share the same one-hot space
        X_train_cat, X_test_cat = X_train_cat.align(
            X_test_cat, join="left", axis=1, fill_value=0
        )
        X_test_cat = X_test_cat[X_train_cat.columns]
    else:
        X_train_cat = pd.DataFrame(index=X_train_raw.index)
        X_test_cat = pd.DataFrame(index=X_test_raw.index)

    # Combine numeric + categorical
    X_train = pd.concat([X_train_num, X_train_cat], axis=1)
    X_test = pd.concat([X_test_num, X_test_cat], axis=1)

    # Reattach label + sensitive attributes as columns for downstream fairness tooling
    train = X_train.copy()
    train[target_col] = y.loc[idx_train].values
    for c in sensitive_cols:
        train[c] = sensitive.loc[idx_train, c].values

    test = X_test.copy()
    test[target_col] = y.loc[idx_test].values
    for c in sensitive_cols:
        test[c] = sensitive.loc[idx_test, c].values

    meta = {
        "target_col": target_col,
        "sensitive_cols": sensitive_cols,
        "n_features": X_train.shape[1],
        "n_train": len(train),
        "n_test": len(test),
        "num_cols": num_cols,
        "cat_cols": cat_cols,
    }
    return train, test, meta

# %%
# German Credit
# Drop the raw personal_status_sex (already decoded into 'sex')
german_train, german_test, german_meta = preprocess_dataset(
    df_german,
    target_col="target",
    sensitive_cols=["sex", "age_group"],
    drop_cols=["personal_status_sex"],
)
print("German train/test:", german_train.shape, german_test.shape)
print("features:", german_meta["n_features"])

# %%
# Taiwan Credit Card Default
# Keep engineered sensitive cols; drop the raw SEX/EDUCATION codes to avoid duplication
taiwan_train, taiwan_test, taiwan_meta = preprocess_dataset(
    df_taiwan,
    target_col="target",
    sensitive_cols=["sex", "education"],
    drop_cols=["SEX", "EDUCATION"],
)
print("Taiwan train/test:", taiwan_train.shape, taiwan_test.shape)
print("features:", taiwan_meta["n_features"])

# %%
# Credit Card Approval
# ID is an identifier, not a feature. Keep both target variants out of the feature matrix.
# The alternate-threshold label is carried alongside for later sensitivity analysis.
home_alt_target = df_home[["target_60dpd"]].copy()  # keep for reference

home_train, home_test, home_meta = preprocess_dataset(
    df_home,
    target_col="target",  # equals target_30dpd (primary)
    sensitive_cols=["sex"],
    drop_cols=["ID", "CODE_GENDER", "target_30dpd", "target_60dpd", "domain"],
)
# Attach the 60dpd label to the matching rows for optional threshold sensitivity checks
home_train = home_train.join(home_alt_target, how="left")
home_test = home_test.join(home_alt_target, how="left")

print("Credit Card Approval train/test:", home_train.shape, home_test.shape)
print("features:", home_meta["n_features"])

# %%
# Save processed splits
german_train.to_parquet(os.path.join(PROCESSED_DIR, "german_train.parquet"))
german_test.to_parquet(os.path.join(PROCESSED_DIR, "german_test.parquet"))
taiwan_train.to_parquet(os.path.join(PROCESSED_DIR, "taiwan_train.parquet"))
taiwan_test.to_parquet(os.path.join(PROCESSED_DIR, "taiwan_test.parquet"))
home_train.to_parquet(os.path.join(PROCESSED_DIR, "credit_card_approval_train.parquet"))
home_test.to_parquet(os.path.join(PROCESSED_DIR, "credit_card_approval_test.parquet"))

# Save metadata for downstream notebooks
metadata = {
    "german": german_meta,
    "taiwan": taiwan_meta,
    "credit_card_approval": home_meta,
    "config": {"random_state": RANDOM_STATE, "test_size": TEST_SIZE},
}
with open(os.path.join(PROCESSED_DIR, "preprocess_meta.json"), "w") as f:
    json.dump(metadata, f, indent=2)

# %%
# Preprocessing summary table
prep_summary = pd.DataFrame([
    {"dataset": "German Credit", "n_features": german_meta["n_features"],
     "n_train": german_meta["n_train"], "n_test": german_meta["n_test"],
     "sensitive": ", ".join(german_meta["sensitive_cols"])},
    {"dataset": "Taiwan CC Default", "n_features": taiwan_meta["n_features"],
     "n_train": taiwan_meta["n_train"], "n_test": taiwan_meta["n_test"],
     "sensitive": ", ".join(taiwan_meta["sensitive_cols"])},
    {"dataset": "Credit Card Approval", "n_features": home_meta["n_features"],
     "n_train": home_meta["n_train"], "n_test": home_meta["n_test"],
     "sensitive": ", ".join(home_meta["sensitive_cols"])},
])
print(prep_summary.to_string(index=False))
prep_summary.to_csv(os.path.join(TAB_DIR, "02_preprocess_summary.csv"), index=False)

german: (1000, 23)
taiwan: (30000, 26)
credit_card_approval: (36457, 22)
German train/test: (750, 60) (250, 60)
features: 57
Taiwan train/test: (22500, 24) (7500, 24)
features: 21
Credit Card Approval train/test: (27342, 56) (9115, 56)
features: 53
             dataset  n_features  n_train  n_test      sensitive
       German Credit          57      750     250 sex, age_group
   Taiwan CC Default          21    22500    7500 sex, education
Credit Card Approval          53    27342    9115            sex
